# SemEval 2027 · DeBERTa-v3-base (P7) — phiên bản đã sửa

**Giữ nguyên bài thử:** Subtask **B**, đầu vào `comment`, `max_length=128`, **Fold 0**, **1 epoch**.

**Đã xử lý trong notebook này:**
- Cố định `transformers` ở nhánh 4.x để tránh lỗi phiên dịch nhầm `spm.model` (SentencePiece) thành `tiktoken` ở bản 5.20.2.
- Dùng bộ tách văn bản **DebertaV2Tokenizer** gốc (`use_fast=False`).
- Chọn **BF16 nếu GPU hỗ trợ**, nếu không thì **FP32** thay vì FP16 (mục tiêu: giảm nguy cơ `NaN`).
- Bỏ các kết quả chạy và nhật ký cài đặt cũ, để Notebook chạy lại từ đầu dễ hơn.

> **Quan trọng:** Chạy ô 1 để cài gói. Sau đó **Restart Kernel / Khởi động lại nhân Python**, rồi chạy **từ ô 2** trở đi. Không chạy lại ô cài đặt giữa lúc đang huấn luyện. Việc huấn luyện cần máy có GPU NVIDIA được PyTorch nhận diện.


In [ ]:
# Ô 1 — CHẠY MỘT LẦN, SAU ĐÓ RESTART KERNEL.
# KHÔNG dùng: %pip install -U transformers (sẽ cài lại bản 5.x gây lỗi spm.model).
# Giữ nguyên PyTorch + CUDA đang cài, chỉ đồng bộ các thư viện xử lý mô hình/văn bản.
%pip install --upgrade "transformers==4.57.6" "huggingface-hub>=0.34.0,<1.0" "sentencepiece>=0.2.0" "protobuf>=4.25,<7" "accelerate>=1.0,<2.0"


### Khởi động lại Kernel trước khi chạy tiếp

Trong VS Code / Jupyter: bấm **Restart Kernel** (hoặc **Khởi động lại**). Nếu không khởi động lại, Python có thể vẫn giữ bản `transformers 5.20.2` trong bộ nhớ dù đã cài 4.57.6.

In [ ]:
# Ô 2 — KIỂM TRA ĐÚNG MÔI TRƯỜNG, NẠP MÃ DỰ ÁN
import sys
from pathlib import Path
import numpy as np
import torch
import transformers

EXPECTED_TRANSFORMERS = "4.57.6"
print("Python:", sys.version.split()[0])
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("Python executable:", sys.executable)

if transformers.__version__ != EXPECTED_TRANSFORMERS:
    raise RuntimeError(
        f"Đang dùng transformers {transformers.__version__}, cần {EXPECTED_TRANSFORMERS}. "
        "Hãy chạy ô 1, chọn Restart Kernel rồi chạy lại từ ô 2."
    )

# Ưu tiên đúng đường dẫn có trong Notebook P7 gốc.
SRC_DIR = Path(r"C:\Users\Public\Documents\NĂM 3\NLP\Đồ án\SemEval2027_Task8\src")
if not (SRC_DIR / "sqe").is_dir():
    # Trường hợp bạn lưu notebook ngay trong dự án hoặc thư mục notebooks/.
    options = [Path.cwd() / "src", Path.cwd().parent / "src"]
    SRC_DIR = next((p for p in options if (p / "sqe").is_dir()), SRC_DIR)

if not (SRC_DIR / "sqe").is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục sqe bên trong {SRC_DIR}. "
        "Hãy sửa biến SRC_DIR để trỏ tới thư mục src của dự án SemEval2027_Task8."
    )

sys.path.insert(0, str(SRC_DIR))
import sqe.train as train_module
from sqe.data import load_en
from sqe.train import DEFAULT, ENCODERS, encode, fit
from transformers import AutoTokenizer
print("Source:", SRC_DIR)
print("Đã nạp thư viện và mã dự án thành công.")


In [ ]:
# Ô 3 — NẠP TOKENIZER SENTENCEPIECE ĐÚNG CÁCH, KIỂM TRA TRƯỚC KHI TRAIN
# Giữ nguyên cấu hình gốc P7: comment, 128 token, 1 epoch, Fold 0.
df = load_en()
cfg = {**DEFAULT, **ENCODERS["deberta"], "input": "comment", "max_length": 128, "epochs": 1}
print("Model:", cfg["model"])
print("Số mẫu:", len(df))

# DeBERTa-v3 dùng SentencePiece; ở Transformers 4.x ép dùng tokenizer gốc.
tok = AutoTokenizer.from_pretrained(cfg["model"], use_fast=False)
print("Tokenizer:", type(tok).__name__, "| is_fast:", tok.is_fast)
if "deberta" in cfg["model"].lower() and (tok.is_fast or type(tok).__name__ != "DebertaV2Tokenizer"):
    raise RuntimeError(
        "Chưa nạp đúng DebertaV2Tokenizer gốc. Kiểm tra transformers==4.57.6 "
        "và sentencepiece; sau đó Restart Kernel."
    )

probe = tok("A brief YouTube comment.", truncation=True, max_length=128)
assert len(probe["input_ids"]) > 0, "Tokenizer chưa tạo được token!"
print("Kiểm tra tokenizer thành công; token mẫu:", len(probe["input_ids"]))

ids = encode(tok, df, cfg)
tr, va = np.where(df.fold != 0)[0], np.where(df.fold == 0)[0]
assert len(tr) > 0 and len(va) > 0, "Chia fold rỗng; cần kiểm tra dữ liệu."
print("Train samples:", len(tr), "| Validation samples:", len(va))


In [ ]:
# Ô 4 — CHỌN BF16 NẾU HỖ TRỢ, CÒN LẠI FP32 (TRÁNH FP16 NaN)
if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch chưa nhận GPU NVIDIA (torch.cuda.is_available() == False). "
        "Hãy kiểm tra trình điều khiển GPU và bản PyTorch có CUDA."
    )

dev = torch.device("cuda")
supports_bf16 = bool(torch.cuda.is_bf16_supported())
safe_dtype = torch.bfloat16 if supports_bf16 else torch.float32

# fit() được định nghĩa trong sqe.train: gán lại hàm của CHÍNH mô-đun để
# các lời gọi _amp_dtype(dev) BÊN TRONG fit() cùng dùng lựa chọn an toàn.
# FP32 có thể khiến torch.autocast cảnh báo rồi tự tắt autocast: đó là hành vi
# dự kiến, cho phép phép toán chạy FP32; tốc độ có thể chậm hơn FP16.
def _safe_amp_dtype(device):
    return torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

train_module._amp_dtype = _safe_amp_dtype
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA hỗ trợ BF16:", supports_bf16)
print("Chế độ chọn:", train_module._amp_dtype(dev))
if safe_dtype == torch.float32:
    print("GPU không hỗ trợ BF16 → ưu tiên FP32 để giảm nguy cơ NaN khi huấn luyện DeBERTa-v3.")


In [ ]:
# Ô 5 — HUẤN LUYỆN 1 FOLD × 1 EPOCH VÀ IN KẾT QUẢ
# Chỉ chạy sau khi cả 4 ô trước đã hoàn tất mà không có lỗi.
model, hist = fit(df, tr, cfg, ("B",), ids, tok, dev, eval_idx=va)
print("Đã kết thúc lời gọi fit().")
print("Lịch sử / chỉ số đánh giá:")
print(hist)

# Nếu chỉ số có NaN/Inf, cần kiểm tra thêm logic hàm fit() trong src/sqe/train.py;
# không thể khẳng định hết NaN chỉ bằng việc sửa cách chọn dtype ở Notebook.
